In [397]:
import pandas as pd
from numexpr.necompiler import double
from sklearn.model_selection import train_test_split
import numpy as np


from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor, Lars
from sklearn.linear_model import LassoLars, OrthogonalMatchingPursuit, PassiveAggressiveRegressor, RANSACRegressor
from sklearn.linear_model import SGDRegressor, TheilSenRegressor, ARDRegression, TweedieRegressor, PoissonRegressor
from sklearn.linear_model import GammaRegressor, QuantileRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, BaggingRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR, LinearSVR, NuSVR
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.dummy import DummyRegressor

In [398]:
models = [
    LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor, Lars,
    LassoLars, OrthogonalMatchingPursuit, PassiveAggressiveRegressor, RANSACRegressor,
    SGDRegressor, TheilSenRegressor, ARDRegression, TweedieRegressor, PoissonRegressor,
    GammaRegressor, QuantileRegressor,
    RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, BaggingRegressor,
    ExtraTreesRegressor, HistGradientBoostingRegressor,
    DecisionTreeRegressor,
    SVR, LinearSVR, NuSVR,
    KNeighborsRegressor, RadiusNeighborsRegressor,
    GaussianProcessRegressor,
    PLSRegression,
    DummyRegressor
]

In [399]:
df = pd.read_csv("Carbon Emission.csv")

In [400]:
df.head()

,Body Type,Sex,Diet,How Often Shower,Heating Energy Source,Transport,Vehicle Type,Social Activity,Monthly Grocery Bill,Frequency of Traveling by Air,Vehicle Monthly Distance Km,Waste Bag Size,Waste Bag Weekly Count,How Long TV PC Daily Hour,How Many New Clothes Monthly,How Long Internet Daily Hour,Energy efficiency,Recycling,Cooking_With,CarbonEmission
0,overweight,female,pescatarian,daily,coal,public,NaN,often,230,frequently,210,large,4,7,26,1,No,['Metal'],"['Stove', 'Oven']",2238
1,obese,female,vegetarian,less frequently,natural gas,walk/bicycle,NaN,often,114,rarely,9,extra large,3,9,38,5,No,['Metal'],"['Stove', 'Microwave']",1892
2,overweight,male,omnivore,more frequently,wood,private,petrol,never,138,never,2472,small,1,14,47,6,Sometimes,['Metal'],"['Oven', 'Microwave']",2595
3,overweight,male,omnivore,twice a day,wood,walk/bicycle,NaN,sometimes,157,rarely,74,medium,3,20,5,7,Sometimes,"['Paper', 'Plastic', 'Glass', 'Metal']","['Microwave', 'Grill', 'Airfryer']",1074
4,obese,female,vegetarian,daily,coal,private,diesel,often,266,very frequently,8457,large,1,3,5,6,Yes,['Paper'],['Oven'],4743


In [401]:
df = df.dropna()

In [402]:
df.reset_index(inplace=True)
df = df.drop(["index"], axis=1)


In [403]:
df.columns

Index(['Body Type', 'Sex', 'Diet', 'How Often Shower', 'Heating Energy Source',
       'Transport', 'Vehicle Type', 'Social Activity', 'Monthly Grocery Bill',
       'Frequency of Traveling by Air', 'Vehicle Monthly Distance Km',
       'Waste Bag Size', 'Waste Bag Weekly Count', 'How Long TV PC Daily Hour',
       'How Many New Clothes Monthly', 'How Long Internet Daily Hour',
       'Energy efficiency', 'Recycling', 'Cooking_With', 'CarbonEmission'],
      dtype='object')

In [404]:
type(df[df.columns[1]][0])

str

In [405]:
mappedColumns = list(df.columns)
mappedColumns.remove("Recycling")
mappedColumns.remove("Cooking_With")

for column in mappedColumns:
    if not isinstance(df[column][0], (int, float, np.number)):
        df[column], unique = pd.factorize(df[column])


In [406]:
recyclingDict = {
    "Metal" : 100,
    "Paper" : 70,
    "Plastic" : 50,
    "Glass" : 40
}

cookingWithDict = {
    "Microwave" : 10,
    "Airfryer" : 20,
    "Stove" : 30,
    "Oven" : 50,
    "Grill" : 120
}

def removeAll(myList, remove):
    newList = []
    for elementIndx in range(0, len(myList) - 1):
        if not ord(myList[elementIndx]) in remove:
            newList.append(myList[elementIndx])
    return newList

def convertLists(column, newColumn, pointMap):
    global df
    df[newColumn] = np.nan
    for index in df.index:
        recycling = list(df[column][index])
        # elements are the ascii for: [ ' ] ,
        removeelements = [91, 39, 93, 44]
        recycling = removeAll(recycling, removeelements)
        recycling = "".join(recycling)
        recycling = recycling.split(" ")
        score = 0
        if recycling[0] != "":
            for material in recycling:
                score += pointMap[material]
        df.loc[index, newColumn] = score

    df = df.drop([column], axis = 1)

convertLists("Recycling", "Recycling Score", recyclingDict)
convertLists("Cooking_With", "Cooking With Score", cookingWithDict)


In [407]:
df

,Body Type,Sex,Diet,How Often Shower,Heating Energy Source,Transport,Vehicle Type,Social Activity,Monthly Grocery Bill,Frequency of Traveling by Air,Vehicle Monthly Distance Km,Waste Bag Size,Waste Bag Weekly Count,How Long TV PC Daily Hour,How Many New Clothes Monthly,How Long Internet Daily Hour,Energy efficiency,CarbonEmission,Recycling Score,Cooking With Score
0,0,0,0,0,0,0,0,0,138,0,2472,0,1,14,47,6,0,2595,100.0,60.0
1,1,1,1,1,1,0,1,1,266,1,8457,1,1,3,5,6,1,4743,70.0,50.0
2,2,1,2,2,0,0,2,0,56,2,5363,2,4,9,11,19,0,1832,0.0,140.0
3,2,1,2,2,2,0,3,2,111,2,2893,1,6,13,16,10,0,1732,190.0,230.0
4,2,0,2,0,2,0,0,1,126,1,7622,2,2,6,37,9,0,5272,0.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3274,3,1,1,0,0,0,0,0,89,2,4482,1,5,15,17,22,0,2999,0.0,60.0
3275,1,0,0,3,1,0,2,2,230,0,268,2,5,12,27,9,1,2408,0.0,10.0
3276,3,1,2,3,1,0,3,0,234,3,5316,3,3,14,8,24,0,3084,120.0,40.0
3277,2,0,2,0,1,0,0,1,179,2,8688,2,5,19,14,5,0,4574,170.0,180.0


In [408]:
#----------------------------------------------Testing---------------------------------------------------

In [409]:
features = df.drop(["CarbonEmission"], axis=1)

In [410]:
features.columns

Index(['Body Type', 'Sex', 'Diet', 'How Often Shower', 'Heating Energy Source',
       'Transport', 'Vehicle Type', 'Social Activity', 'Monthly Grocery Bill',
       'Frequency of Traveling by Air', 'Vehicle Monthly Distance Km',
       'Waste Bag Size', 'Waste Bag Weekly Count', 'How Long TV PC Daily Hour',
       'How Many New Clothes Monthly', 'How Long Internet Daily Hour',
       'Energy efficiency', 'Recycling Score', 'Cooking With Score'],
      dtype='object')

In [411]:
classifier = df["CarbonEmission"]

In [412]:
classifier

0       2595
1       4743
2       1832
3       1732
4       5272
        ... 
3274    2999
3275    2408
3276    3084
3277    4574
3278     826
Name: CarbonEmission, Length: 3279, dtype: int64

In [414]:
scoring = {}

for model in models:
    # fit model
    score = 0
    for i in range(1):
        x_train, x_test, y_train, y_test = train_test_split(features, classifier)
        bot = model()
        bot.fit(x_train, y_train)
        score += bot.score(x_test, y_test)
    scoring[model.__name__] = score

    # test model
    # Display results

C:\Users\kb148\anaconda3\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
C:\Users\kb148\anaconda3\Lib\site-packages\sklearn\linear_model\_glm\glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)
C:\Users\kb148\anaconda3\Lib\site-packages\sklearn\linear_model\_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
C:\Users\kb148\an

In [421]:
scoring

{'LinearRegression': 0.5896322186186943,
 'Ridge': 0.5478970213570258,
 'Lasso': 0.5575197694208912,
 'ElasticNet': 0.5564005616806549,
 'BayesianRidge': 0.5856883388960599,
 'HuberRegressor': 0.415435619708025,
 'Lars': 0.5745157685340838,
 'LassoLars': 0.5836721012437955,
 'OrthogonalMatchingPursuit': 0.27538074029804704,
 'PassiveAggressiveRegressor': -8.950896124757834,
 'RANSACRegressor': 0.39236032915041874,
 'SGDRegressor': -2.4150391290330398e+26,
 'TheilSenRegressor': 0.5545926601997802,
 'ARDRegression': 0.5803241124616969,
 'TweedieRegressor': np.float64(0.4889221265112508),
 'PoissonRegressor': np.float64(-6.536092779096947e-05),
 'GammaRegressor': np.float64(0.45115617753210613),
 'QuantileRegressor': 0.31063844425110554,
 'RandomForestRegressor': 0.8613667098500593,
 'GradientBoostingRegressor': 0.9386582218376319,
 'AdaBoostRegressor': 0.7592564200209748,
 'BaggingRegressor': 0.8591693678591148,
 'ExtraTreesRegressor': 0.8942487802796555,
 'HistGradientBoostingRegressor'

In [427]:
scoringData = pd.Series(scoring)
scoringData.sort_values(ascending=False, inplace=True)

In [428]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [429]:
scoringData

HistGradientBoostingRegressor                                   0.96
GradientBoostingRegressor                                       0.94
ExtraTreesRegressor                                             0.89
RandomForestRegressor                                           0.86
BaggingRegressor                                                0.86
AdaBoostRegressor                                               0.76
DecisionTreeRegressor                                           0.72
LinearRegression                                                0.59
BayesianRidge                                                   0.59
PLSRegression                                                   0.59
LassoLars                                                       0.58
ARDRegression                                                   0.58
Lars                                                            0.57
Lasso                                                           0.56
ElasticNet                        